# DL_05 — Hybrid AML + DL Surrogate

**Owner: Samarth Sangtani**  
**Work package 5 of 5 — Phase 1/Phase 2 integration and hybrid validation**

## 1. Hybrid idea

Phase 1 already provides an XGBoost surrogate that maps layout-level geometric summaries and wind direction to one canopy-averaged wind-speed scalar. Phase 2 U-Net maps the rasterized layout and conditions to a pedestrian-level spatial velocity field.

The hybrid design uses the Phase 1 XGBoost prediction as an additional **global context channel** repeated over the spatial grid. The U-Net then predicts the local `uped` and `vped` fields using both:

- local spatial information from the building-height raster;
- global flow context from the Phase 1 canopy-mean prediction.

This is a meaningful coarse-to-fine combination: the AML model summarizes global layout response, while the DL model learns spatial redistribution around buildings.

## 2. Important physical distinction

The Phase 1 target is canopy-averaged wind speed from ground to mean building height. The Phase 2 target is the pair of pedestrian-level component maps from the `*_ped.nc` files. These are not the same quantity. Therefore, the hybrid does **not** force the average predicted pedestrian field to equal the Phase 1 canopy scalar. The XGBoost scalar is used only as a conditioning feature, not as a pixel-wise target or a consistency equation.

## 3. Leakage prevention

- For Phase 2 training cases, the Phase 1 XGBoost channel is generated using family-grouped out-of-fold predictions. Each training case's own scalar target is excluded from the XGBoost model that predicts its conditioning value.
- Validation and test scalar predictions come from an XGBoost model fitted only on Phase 2 training families.
- Family/layout identifiers are used for splitting only, never as XGBoost predictors.
- The Phase 2 test families remain locked.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Locate the existing repository before importing the shared Phase 2 package.
current = Path.cwd().resolve()
candidates = [current, *current.parents]
ROOT = None
for candidate in candidates:
    if (candidate / "data" / "raw").is_dir() and (candidate / "Machine Learning Pipeline").is_dir():
        ROOT = candidate
        break
    if candidate.is_dir():
        for child in candidate.iterdir():
            if child.is_dir() and (child / "data" / "raw").is_dir() and (child / "Machine Learning Pipeline").is_dir():
                ROOT = child
                break
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError("Clone/open the existing T006 repository first; the Phase 2 folder must be added at its root.")

sys.path.insert(0, str(ROOT))
from dl_src.paths import get_project_paths
PATHS = get_project_paths(ROOT)
DL_RESULTS = PATHS["dl_results"]
FIELDS_DIR = PATHS["fields"]
FIGURES_DIR = PATHS["figures"]
MODELS_DIR = PATHS["models"]
PREDICTIONS_DIR = PATHS["predictions"]

print("Repository root:", ROOT)
print("Runtime outputs:", DL_RESULTS)


In [ ]:
import torch
from torch.utils.data import DataLoader
from dl_src.data import FlowFieldDataset
from dl_src.models import UNetVelocity, load_unet_checkpoint
from dl_src.training import train_model
from dl_src.metrics import evaluate_model
from dl_src.hybrid import fit_leakage_safe_canopy_predictions

from dl_src.training import get_torch_device
DEVICE = get_torch_device()
manifest = pd.read_csv(DL_RESULTS / "dl_case_metadata_manifest.csv")
split_manifest = pd.read_csv(DL_RESULTS / "dl_split_manifest.csv")
with open(DL_RESULTS / "normalization_stats.json", "r", encoding="utf-8") as file:
    normalization_stats = json.load(file)

feature_engineered_path = PATHS["processed"] / "feature_engineered_dataset.csv"
print("Phase 1 engineered table:", feature_engineered_path)
if not feature_engineered_path.exists():
    raise FileNotFoundError(
        "Run the existing Phase 1 ML_01 and ML_03 notebooks first. "
        "The hybrid uses the Phase 1 feature-engineered table and does not recreate it."
    )

## 4. Generate leakage-safe Phase 1 scalar predictions

The Phase 1 XGBoost feature list and fixed hyperparameters are recreated from the existing ML_06 notebook. Grouped out-of-fold predictions are generated for Phase 2 training cases. This avoids feeding an overly optimistic in-sample prediction into the hybrid U-Net.

In [ ]:
prediction_lookup, canopy_prediction_stats, phase1_prediction_table, phase1_scalar_metrics = fit_leakage_safe_canopy_predictions(
    feature_engineered_path=feature_engineered_path,
    split_manifest=split_manifest,
    results_dir=DL_RESULTS,
)

print("Phase 1 XGBoost test metrics (scalar target only):")
print(json.dumps(phase1_scalar_metrics, indent=2))
display(phase1_prediction_table.head())
print("Number of case-level scalar predictions:", len(prediction_lookup))

## 5. Create hybrid data loaders

The sixth input channel is the standardized Phase 1 XGBoost canopy-mean prediction. Its normalization statistics are fitted from training OOF predictions only. The original five raster/condition channels remain unchanged.

In [ ]:
hybrid_train_dataset = FlowFieldDataset(
    manifest, split_manifest, FIELDS_DIR, "train", normalization_stats,
    canopy_predictions=prediction_lookup,
    canopy_prediction_stats=canopy_prediction_stats,
)
hybrid_validation_dataset = FlowFieldDataset(
    manifest, split_manifest, FIELDS_DIR, "validation", normalization_stats,
    canopy_predictions=prediction_lookup,
    canopy_prediction_stats=canopy_prediction_stats,
)
hybrid_test_dataset = FlowFieldDataset(
    manifest, split_manifest, FIELDS_DIR, "test", normalization_stats,
    canopy_predictions=prediction_lookup,
    canopy_prediction_stats=canopy_prediction_stats,
)

hybrid_train_loader = DataLoader(hybrid_train_dataset, batch_size=1, shuffle=True, num_workers=0)
hybrid_validation_loader = DataLoader(hybrid_validation_dataset, batch_size=1, shuffle=False, num_workers=0)
hybrid_test_loader = DataLoader(hybrid_test_dataset, batch_size=1, shuffle=False, num_workers=0)

print("Hybrid input shape:", tuple(hybrid_train_dataset[0][0].shape))
print("Expected channels: 6 = 5 original channels + Phase 1 XGBoost global context")

## 6. Train the hybrid conditional U-Net

The hybrid U-Net has the same spatial architecture and optimization settings as the base U-Net, except for the additional global scalar channel. This keeps the comparison interpretable.

In [ ]:
HYBRID_CONFIG = {
    "in_channels": 6,
    "out_channels": 2,
    "base_channels": 8,
    "dropout": 0.10,
}

hybrid_model = UNetVelocity(**HYBRID_CONFIG)
hybrid_model, hybrid_history, hybrid_checkpoint = train_model(
    model=hybrid_model,
    train_loader=hybrid_train_loader,
    validation_loader=hybrid_validation_loader,
    device=DEVICE,
    output_dir=DL_RESULTS,
    model_name="hybrid_unet",
    epochs=25,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=7,
    gradient_clip=1.0,
    model_config=HYBRID_CONFIG,
)
display(hybrid_history.tail())
print("Hybrid checkpoint:", hybrid_checkpoint)

## 7. Evaluate base U-Net and hybrid on the same test cases

The base U-Net and hybrid are evaluated with identical target fields, valid masks, family-held-out test cases and metric definitions. This is the direct test of whether the AML global-context channel adds value.

In [ ]:
hybrid_case_metrics, hybrid_pooled_metrics, hybrid_predictions = evaluate_model(
    model=hybrid_model,
    data_loader=hybrid_test_loader,
    normalization_stats=normalization_stats,
    device=DEVICE,
    model_name="hybrid_unet",
    results_dir=DL_RESULTS,
    fields_dir=FIELDS_DIR,
)

base_model, base_checkpoint = load_unet_checkpoint(MODELS_DIR / "unet_best.pt", DEVICE)
base_test_dataset = FlowFieldDataset(
    manifest, split_manifest, FIELDS_DIR, "test", normalization_stats
)
base_test_loader = DataLoader(base_test_dataset, batch_size=1, shuffle=False, num_workers=0)
base_case_metrics, base_pooled_metrics, base_predictions = evaluate_model(
    model=base_model,
    data_loader=base_test_loader,
    normalization_stats=normalization_stats,
    device=DEVICE,
    model_name="unet_rechecked_for_hybrid_comparison",
    results_dir=DL_RESULTS,
    fields_dir=FIELDS_DIR,
)

print("Base U-Net pooled field metrics:")
display(base_pooled_metrics)
print("Hybrid U-Net pooled field metrics:")
display(hybrid_pooled_metrics)

## 8. Quantify the hybrid contribution

The field metrics compare base U-Net and hybrid U-Net. The Phase 1 XGBoost scalar MAE is reported separately because it is a different target and has different units. Do not place the scalar MAE beside field MAE as if they measured the same task.

In [ ]:
base_row = base_pooled_metrics.iloc[0].to_dict()
hybrid_row = hybrid_pooled_metrics.iloc[0].to_dict()
comparison = pd.DataFrame([
    {"model": "Phase 2 U-Net", **base_row},
    {"model": "Hybrid XGBoost-conditioned U-Net", **hybrid_row},
])
comparison.to_csv(DL_RESULTS / "hybrid_vs_base_unet_field_metrics.csv", index=False)
with open(DL_RESULTS / "hybrid_phase1_scalar_test_metrics.json", "r", encoding="utf-8") as file:
    scalar_metrics = json.load(file)

print("Field-model comparison:")
display(comparison)
print("Separate Phase 1 scalar test metrics:")
print(json.dumps(scalar_metrics, indent=2))

base_mae = float(base_row["field_component_MAE"])
hybrid_mae = float(hybrid_row["field_component_MAE"])
if base_mae > 0:
    relative_change = 100.0 * (base_mae - hybrid_mae) / base_mae
    print(f"Change in pooled field MAE after adding the AML context channel: {relative_change:.2f}%")
    if relative_change > 0:
        print("The hybrid reduced the measured field MAE on this held-out test set.")
    elif relative_change < 0:
        print("The hybrid did not improve field MAE on this test set; report this honestly and discuss possible reasons.")
    else:
        print("The hybrid and base U-Net have equal pooled field MAE on this test set.")

## 9. Hybrid interpretation and limitations

The hybrid is technically justified because it combines two different information scales: Phase 1 supplies a global scalar summary, and Phase 2 predicts local spatial structure. The training channel is leakage-safe because training values are out-of-fold predictions.

The hybrid may or may not improve the test metric. A technically sound negative result is still reportable; do not tune repeatedly against the test set or claim improvement if the measured field error does not improve. If the hybrid does not help, discuss whether the Phase 1 scalar contains information already recoverable from the raster/conditions, whether the small family count limits the experiment, and whether the scalar target's canopy-average definition differs from the pedestrian-level field target.

## DL_05 completion checklist

- [ ] Phase 1 XGBoost uses the original Phase 1 predictor whitelist and fixed parameters.
- [ ] Training scalar conditioning values are family-grouped out-of-fold predictions.
- [ ] Validation/test scalar values are predicted by a model fitted only on training families.
- [ ] Hybrid model adds one global context channel and predicts the same two spatial components.
- [ ] Base and hybrid U-Nets are evaluated on exactly the same test cases.
- [ ] Field and scalar metrics are kept separate.
- [ ] The report states the hybrid's measured benefit or lack of benefit honestly.

**Final handoff:** the five owners combine the generated `dl_results/` tables, plots and checkpoints into the Phase 2 report, presentation and viva evidence.